In [102]:
# ==========================================
# 1. CONFIGURATION & GLOBAL DATA LOAD
# ==========================================
import os
import re
import json
import numpy as np
import pandas as pd
import open3d as o3d
import matplotlib.pyplot as plt

# File paths and naming
EXPERIMENT = "training_second_square_flange"
WORKPIECE_NAME = "workpiece3"

WORKPIECE_PATH = f"workpiece/{WORKPIECE_NAME}/workpiece.stl"
POSES_PATH = f"viewpoints_candidate/testing_data/{EXPERIMENT}"
CSV_PATH = f"surface/{EXPERIMENT}/chamfer_results.csv"

# The EXACT 40k points saved from 1_3_viewpoint_generation_manual
GLOBAL_PCD_PATH = f"viewpoints_candidate/testing_data/{EXPERIMENT}/pcd_all.pcd"
GLOBAL_COVERED_JSON = f"viewpoints_candidate/testing_data/{EXPERIMENT}/covered_indices.json"

# Optimization Settings
OPTIMIZATION_METHOD = "GRASP"  # "GREEDY" or "GRASP"
GRASP_ITERATIONS = 10          # Number of sequences to generate
RCL_SIZE = 5                   # Top N candidates to pick randomly from (Cardinality-based RCL)

# Utility weights & Discretization
ALPHA = 0.75
BETA = 0.25
GAMMA = 0.5  # Submodular decay factor for Coverability
TOTAL_STEPS = 8  # Number of viewpoints to select
DISTANCE_THRESHOLD = 1.0  # mm
AXIS_SIZE = 20.0
NUM_BINS = 4

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def make_arrow(direction='x', size=50.0, color=(0.6, 0.6, 0.6)):
    arrow = o3d.geometry.TriangleMesh.create_arrow(
        cylinder_radius=size * 0.05,
        cone_radius=size * 0.15,
        cylinder_height=size * 0.80,
        cone_height=size * 0.20,
        resolution=20,
    )
    if direction == 'x':
        R_align = arrow.get_rotation_matrix_from_xyz((0, np.pi / 2, 0))
        arrow.rotate(R_align, center=(0, 0, 0))
    arrow.paint_uniform_color(list(color))
    arrow.compute_vertex_normals()
    return arrow

def make_xz_arrows(transform=None, size=50.0, color=(0.6, 0.6, 0.6)):
    frame = make_arrow('x', size=size, color=color) + make_arrow('z', size=size, color=color)
    if transform is not None:
        frame.transform(transform)
    return frame

def load_viewpoint_poses_dict(folder_path):
    def extract_number(filename):
        match = re.search(r'viewpoint_pose_(\d+)\.npy', filename)
        return int(match.group(1)) if match else -1
    
    npy_files = [f for f in os.listdir(folder_path) if f.endswith('.npy')]
    poses = {}
    for f in npy_files:
        idx = extract_number(f)
        if idx != -1:
            poses[idx] = np.load(os.path.join(folder_path, f))
    return poses

# ==========================================
# 3. LOAD GLOBAL DATA & MULTIPLE FEATURES
# ==========================================
print("Loading Global 40k Point Cloud...")
pcd_all = o3d.io.read_point_cloud(GLOBAL_PCD_PATH) if os.path.exists(GLOBAL_PCD_PATH) else o3d.geometry.PointCloud()
pcd_all.paint_uniform_color([0.6, 0.6, 0.6])

print("Loading Global Viewpoint Visibilities...")
if os.path.exists(GLOBAL_COVERED_JSON):
    with open(GLOBAL_COVERED_JSON, "r") as f:
        global_visibility_dict = json.load(f)
else:
    global_visibility_dict = {}

poses_dict = load_viewpoint_poses_dict(POSES_PATH)
df_chamfer = pd.read_csv(CSV_PATH)
features = df_chamfer['Feature'].unique()

print(f"\nFound {len(features)} unique features to optimize: {features}")

feature_indices = {}
all_features_indices = set()

for feat in features:
    feat_path = f"workpiece/{WORKPIECE_NAME}/{feat}.stl"
    if os.path.exists(feat_path):
        f_mesh = o3d.io.read_triangle_mesh(feat_path)
        f_pcd = f_mesh.sample_points_poisson_disk(number_of_points=5000)
        
        dists = np.asarray(pcd_all.compute_point_cloud_distance(f_pcd))
        idx_set = set(np.where(dists < DISTANCE_THRESHOLD)[0])
        feature_indices[feat] = idx_set
        all_features_indices.update(idx_set)
        
        print(f"  - {feat}: {len(idx_set)} matching points in 40k cloud")
    else:
        print(f"  - WARNING: {feat_path} not found!")

print(f"\nTotal target points across all features: {len(all_features_indices)}")


Loading Global 40k Point Cloud...
Loading Global Viewpoint Visibilities...

Found 3 unique features to optimize: ['surface4' 'surface5' 'surface6']
  - surface4: 2694 matching points in 40k cloud
  - surface5: 2089 matching points in 40k cloud
  - surface6: 1398 matching points in 40k cloud

Total target points across all features: 5688


In [103]:
# ==========================================
# 4. OPTIMIZATION RUN (GREEDY / GRASP)
# ==========================================
import pandas as pd
import numpy as np
import random

if OPTIMIZATION_METHOD == "GREEDY":
    num_iterations = 1
    rcl_size = 1
else:
    num_iterations = GRASP_ITERATIONS
    rcl_size = RCL_SIZE

if len(all_features_indices) == 0:
    print("WARNING: all_features_indices is empty. Make sure you loaded the data correctly.")

print(f"Starting {OPTIMIZATION_METHOD} Optimization for {TOTAL_STEPS} steps...")
if OPTIMIZATION_METHOD == "GRASP":
    print(f"Running {num_iterations} iterations with RCL Size = {rcl_size} (Cardinality)...\n")

# Global tracking for the absolute best sequence found
best_overall_sequence = []
best_overall_score = -1.0
best_overall_point_counts = None
best_step_to_covered = {}
best_all_step_results = []

for iteration in range(1, num_iterations + 1):
    
    # State is completely reset at the start of every sequence generation
    point_coverage_counts = np.zeros(len(pcd_all.points))
    selected_viewpoints = []
    step_to_covered_indices = {}
    all_step_results = []
    
    cumulative_utility = 0.0
    
    for k in range(1, TOTAL_STEPS + 1):
        db_records = []
        
        for v_idx, group in df_chamfer.groupby('Viewpoint'):
            v_idx = int(v_idx)
            
            if v_idx not in poses_dict or pd.isna(group['Chamfer_Distance_mm'].iloc[0]):
                continue
            if v_idx in selected_viewpoints:
                continue
                
            camera_visible_40k = set(global_visibility_dict.get(str(v_idx), []))
            
            # --------------------------------------------------
            # 1. Measure Coverability with Submodular Decay (Gamma)
            # --------------------------------------------------
            visible_target_indices = camera_visible_40k & all_features_indices
            if len(visible_target_indices) == 0:
                continue
            
            submodular_sum = 0.0
            for idx in visible_target_indices:
                c = point_coverage_counts[idx]
                submodular_sum += (GAMMA ** c)
                
            coverability = submodular_sum / len(all_features_indices)
            
            # --------------------------------------------------
            # 2. Weighted Chamfer Distance
            # --------------------------------------------------
            sum_weighted_chamfer = 0.0
            sum_points = 0
            
            for _, row in group.iterrows():
                feature = row['Feature']
                chamfer_dist = row['Chamfer_Distance_mm']
                
                if chamfer_dist < 0 or feature not in feature_indices:
                    continue
                    
                P_vj = len(camera_visible_40k & feature_indices[feature])
                sum_weighted_chamfer += (P_vj * chamfer_dist)
                sum_points += P_vj
                
            if sum_points == 0:
                continue
                
            weighted_chamfer = sum_weighted_chamfer / sum_points
            newly_covered = [idx for idx in visible_target_indices if point_coverage_counts[idx] == 0]
            
            db_records.append({
                'Step': k,
                'Viewpoint': v_idx,
                'Coverability': coverability,
                'Chamfer_Distance': weighted_chamfer,
                '_visible_target_indices': visible_target_indices,
                '_new_points_set': newly_covered
            })
        
        df_step = pd.DataFrame(db_records)
        if df_step.empty:
            break
            
        # --- Normalize Metrics ---
        min_cov = df_step['Coverability'].min()
        max_cov = df_step['Coverability'].max()
        df_step['Norm_Coverability'] = (df_step['Coverability'] - min_cov) / (max_cov - min_cov) if max_cov > min_cov else 1.0
        
        # --- Uncertainty & Confidence ---
        df_step['Uncertainty'] = (1.0 * df_step['Norm_Coverability']) + (1.0 * df_step['Chamfer_Distance'])
        min_uncert = df_step['Uncertainty'].min()
        max_uncert = df_step['Uncertainty'].max()
        df_step['Confidence'] = 1.0 - ((df_step['Uncertainty'] - min_uncert) / (max_uncert - min_uncert)) if max_uncert > min_uncert else 1.0
        
        # --- Utility Score ---
        df_step['Information_Gain'] = df_step['Norm_Coverability']
        df_step['Utility_Score'] = ALPHA * df_step['Information_Gain'] + BETA * df_step['Confidence']
        
        # Rank Candidates
        df_step = df_step.sort_values(by='Utility_Score', ascending=False).reset_index(drop=True)
        df_step['Rank'] = df_step.index + 1
        
        # --------------------------------------------------
        # RCL SELECTION (CARDINALITY)
        # --------------------------------------------------
        actual_rcl_size = min(rcl_size, len(df_step))
        rcl = df_step.head(actual_rcl_size)
        
        # Pick completely randomly from the RCL
        chosen_idx = random.randint(0, actual_rcl_size - 1)
        best_row = rcl.iloc[chosen_idx]
        
        best_viewpoint = best_row['Viewpoint']
        best_visible_indices = best_row['_visible_target_indices']
        best_new_points = best_row['_new_points_set']
        
        # Track total utility across the sequence
        cumulative_utility += best_row['Utility_Score']
        
        selected_viewpoints.append(int(best_viewpoint))
        all_step_results.append(df_step.drop(columns=['_visible_target_indices', '_new_points_set']))
        step_to_covered_indices[k] = best_new_points
        
        # Update state counts for the next step in this sequence
        for idx in best_visible_indices:
            point_coverage_counts[idx] += 1
            
    # After generating a full sequence, check if it's the best one we've seen globally
    if cumulative_utility > best_overall_score:
        best_overall_score = cumulative_utility
        best_overall_sequence = selected_viewpoints
        best_overall_point_counts = point_coverage_counts.copy()
        best_step_to_covered = step_to_covered_indices
        best_all_step_results = all_step_results
        
    if OPTIMIZATION_METHOD == "GRASP" and (iteration % 10 == 0 or iteration == num_iterations):
        print(f"Iteration {iteration}/{num_iterations} | Best Sum Utility: {best_overall_score:.4f}")

# Expose the best iteration globally so visualization blocks can map colors correctly
point_coverage_counts = best_overall_point_counts
selected_viewpoints = best_overall_sequence
step_to_covered_indices = best_step_to_covered
all_step_results = best_all_step_results

print(f"\n{'='*40}")
print(f"OPTIMIZATION COMPLETE ({OPTIMIZATION_METHOD})")
print(f"Total points covered at least once: {np.sum(point_coverage_counts > 0)} / {len(all_features_indices)}")
print(f"Best Sequence Sum Utility: {best_overall_score:.4f}")
print(f"Best Viewpoint Sequence: {selected_viewpoints}\n")

if OPTIMIZATION_METHOD == "GREEDY":
    for k, df_step in enumerate(all_step_results, 1):
        print(f"--- STEP {k} Top Candidates ---")
        display(df_step.head(5))
else:
    print("The above sequence is the best sequence found across all GRASP iterations!")


Starting GRASP Optimization for 8 steps...
Running 10 iterations with RCL Size = 5 (Cardinality)...

Iteration 10/10 | Best Sum Utility: 7.5928

OPTIMIZATION COMPLETE (GRASP)
Total points covered at least once: 5686 / 5688
Best Sequence Sum Utility: 7.5928
Best Viewpoint Sequence: [16, 46, 26, 20, 42, 25, 47, 31]

The above sequence is the best sequence found across all GRASP iterations!


In [105]:
# ==========================================
# 3. VISUALIZATION OF STEP-BY-STEP COVERAGE
# ==========================================
import copy
import matplotlib.pyplot as plt

# Create a copy of pcd_all to colorize
vis_pcd = copy.deepcopy(pcd_all)

# Default color (light gray) for unseen points
colors = np.ones((len(vis_pcd.points), 3)) * 0.8

# Generate distinct colors for each step (e.g. from tab10 colormap)
cmap = plt.get_cmap("tab10")

print("Point Colors:")
for k in range(1, TOTAL_STEPS + 1):
    if k in step_to_covered_indices:
        step_color = cmap(k - 1)[:3]  # RGB from colormap
        
        # Color text for the print statement using ANSI escape codes
        r, g, b = [int(c * 255) for c in step_color]
        colored_text = f"\033[38;2;{r};{g};{b}mStep {k}\033[0m"
        print(f"{colored_text} covered {len(step_to_covered_indices[k])} points.")
        
        indices = list(step_to_covered_indices[k])
        colors[indices] = step_color

vis_pcd.colors = o3d.utility.Vector3dVector(colors)

# Draw geometries
print("\nOpening Open3D visualization window...")
o3d.visualization.draw_geometries(
    [vis_pcd], 
    window_name="Step-by-Step Coverage",
    width=1024, height=768,
    front=[0, 0, 1], lookat=[0, 0, 0], up=[0, 1, 0], zoom=1.0
)


Point Colors:
Step 1 covered 4041 points.
Step 2 covered 1284 points.
Step 3 covered 186 points.
Step 4 covered 173 points.
Step 5 covered 2 points.
Step 6 covered 0 points.
Step 7 covered 0 points.
Step 8 covered 0 points.

Opening Open3D visualization window...


In [98]:
# ==========================================
# 4. EXTRA: VISUALIZE ONLY DETECTED POINTS
# ==========================================
import open3d as o3d
import numpy as np

print("Filtering out unseen (grey) points...")

# Collect all indices of points covered in ANY step
all_covered_indices = set()
for step, indices in step_to_covered_indices.items():
    all_covered_indices.update(indices)

# Use Open3D's select_by_index to extract ONLY the colored points
covered_pcd = vis_pcd.select_by_index(list(all_covered_indices))

print(f"Total points shown: {len(covered_pcd.points)}")

# Draw geometries
print("\nOpening Open3D visualization window (Detected Points Only)...")
o3d.visualization.draw_geometries(
    [covered_pcd], 
    window_name="Detected Points Only",
    width=1024, height=768,
    front=[0, 0, 1], lookat=[0, 0, 0], up=[0, 1, 0], zoom=1.0
)


Filtering out unseen (grey) points...
Total points shown: 5681

Opening Open3D visualization window (Detected Points Only)...


In [72]:
# ==========================================
# VISUALIZATION SCRIPT
# ==========================================
import os
import re
import numpy as np
import pandas as pd
import open3d as o3d
import matplotlib.pyplot as plt

# 1. CONFIGURATION
EXPERIMENT = "training_second_square_flange"
FEATURE_NAME = "surface6"
WORKPIECE_NAME = "workpiece3"

WORKPIECE_PATH = "workpiece/workpiece3/workpiece.stl"
FEATURE_PATH = f"workpiece/workpiece3/{FEATURE_NAME}.stl"
POSES_PATH = f"viewpoints_candidate/testing_data/{EXPERIMENT}"
CSV_PATH = f"surface/{EXPERIMENT}/chamfer_results.csv"
AXIS_SIZE = 20.0
NUM_BINS = 4
DISTANCE_THRESHOLD = 1.0  # Threshold to remove overlapping points (mm)

# 2. HELPER FUNCTIONS
def make_arrow(direction='x', size=50.0, color=(0.6, 0.6, 0.6)):
    arrow = o3d.geometry.TriangleMesh.create_arrow(
        cylinder_radius=size * 0.05,
        cone_radius=size * 0.15,
        cylinder_height=size * 0.80,
        cone_height=size * 0.20,
        resolution=20,
    )
    if direction == 'x':
        R_align = arrow.get_rotation_matrix_from_xyz((0, np.pi / 2, 0))
        arrow.rotate(R_align, center=(0, 0, 0))
    arrow.paint_uniform_color(list(color))
    arrow.compute_vertex_normals()
    return arrow

def make_xz_arrows(transform=None, size=50.0, color=(0.6, 0.6, 0.6)):
    frame = make_arrow('x', size=size, color=color) + make_arrow('z', size=size, color=color)
    if transform is not None:
        frame.transform(transform)
    return frame

def visualize_viewpoints_colored(meshes, matrices, colors_per_frame, axis_size=50.0):
    if isinstance(matrices, np.ndarray) and matrices.ndim == 2:
        matrices = [matrices]

    geometries = list(meshes)
    geometries.append(make_xz_arrows(transform=None, size=axis_size, color=(0.9, 0.9, 0.9)))

    for i, (mat, color) in enumerate(zip(matrices, colors_per_frame)):
        geometries.append(make_xz_arrows(transform=mat, size=axis_size, color=color))

    print(f"\nVisualizing {len(matrices)} viewpoint(s) for feature '{FEATURE_NAME}'...")
    # If Open3D fails to show window in your environment, sometimes calling it without kwargs helps.
    o3d.visualization.draw_geometries(
        geometries,
        window_name="Feature Viewpoint Visualization",
        width=1024, height=768,
        front=[0, 0, 1], lookat=[0, 0, 0], up=[0, 1, 0], zoom=1.0
    )

def load_viewpoint_poses_dict(folder_path):
    def extract_number(filename):
        match = re.search(r'viewpoint_pose_(\d+)\.npy', filename)
        return int(match.group(1)) if match else -1
    
    npy_files = [f for f in os.listdir(folder_path) if f.endswith('.npy')]
    poses = {}
    for f in npy_files:
        idx = extract_number(f)
        if idx != -1:
            try:
                poses[idx] = np.load(os.path.join(folder_path, f))
            except Exception as e:
                print(f"Error loading {f}: {e}")
    return poses

# 3. LOAD DATA & COLOR PROCESSING
if os.path.exists(WORKPIECE_PATH):
    workpiece_mesh = o3d.io.read_triangle_mesh(WORKPIECE_PATH)
    workpiece_mesh.compute_vertex_normals()
    print("Sampling points from the whole workpiece mesh to create a PointCloud...")
    workpiece_pcd = workpiece_mesh.sample_points_poisson_disk(number_of_points=10000)
    workpiece_pcd.paint_uniform_color([0.6, 0.6, 0.6])  # Gray for the whole workpiece
else:
    print(f"Warning: {WORKPIECE_PATH} not found.")
    workpiece_pcd = o3d.geometry.PointCloud()

if os.path.exists(FEATURE_PATH):
    feature_mesh = o3d.io.read_triangle_mesh(FEATURE_PATH)
    feature_mesh.compute_vertex_normals()
    print("Sampling points from the feature mesh to create a PointCloud...")
    feature_pcd = feature_mesh.sample_points_poisson_disk(number_of_points=5000)
    feature_pcd.paint_uniform_color([1.0, 0.0, 0.0])  # Red for the feature
else:
    print(f"Warning: {FEATURE_PATH} not found.")
    feature_pcd = o3d.geometry.PointCloud()

# Remove overlapping points from the workpiece so the feature is perfectly clear
if len(workpiece_pcd.points) > 0 and len(feature_pcd.points) > 0:
    print("Removing points from the workpiece that overlap with the selected feature...")
    dists = workpiece_pcd.compute_point_cloud_distance(feature_pcd)
    dists = np.asarray(dists)
    non_feature_indices = np.where(dists > DISTANCE_THRESHOLD)[0]
    workpiece_pcd = workpiece_pcd.select_by_index(non_feature_indices)

poses_dict = load_viewpoint_poses_dict(POSES_PATH)
print(f"Loaded {len(poses_dict)} viewpoint poses from {POSES_PATH}")

df_chamfer = pd.read_csv(CSV_PATH)
df_feature = df_chamfer[df_chamfer['Feature'] == FEATURE_NAME].copy()
if df_feature.empty:
    print(f"No data found for feature '{FEATURE_NAME}' in CSV!")
else:
    print(f"Found {len(df_feature)} rows for feature '{FEATURE_NAME}'.")
    _cmap = plt.get_cmap('coolwarm')
    CLASS_COLORS = {cls: tuple(_cmap(cls / (NUM_BINS - 1))[:3]) for cls in range(NUM_BINS)}
    _, bins = pd.cut(df_feature['Chamfer_Distance_mm'], bins=NUM_BINS, retbins=True)
    df_feature['Error_Class'] = pd.cut(
        df_feature['Chamfer_Distance_mm'], bins=bins, labels=range(NUM_BINS), include_lowest=True
    )

valid_poses = []
valid_classes = []
for _, row in df_feature.iterrows():
    v_idx = int(row['Viewpoint'])
    if v_idx in poses_dict and pd.notna(row['Error_Class']):
        valid_poses.append(poses_dict[v_idx])
        valid_classes.append(int(row['Error_Class']))

colors_per_frame = [CLASS_COLORS[ec] for ec in valid_classes]
if not df_feature.empty:
    print(f"Chamfer Distance range: {df_feature['Chamfer_Distance_mm'].min():.2f} - {df_feature['Chamfer_Distance_mm'].max():.2f} mm")

# 4. SHOW RESULTS
if valid_poses:
    visualize_viewpoints_colored([workpiece_pcd, feature_pcd], valid_poses, colors_per_frame, axis_size=AXIS_SIZE)
else:
    print("No valid viewpoint poses to visualize.")

Sampling points from the whole workpiece mesh to create a PointCloud...
Sampling points from the feature mesh to create a PointCloud...
Removing points from the workpiece that overlap with the selected feature...
Loaded 432 viewpoint poses from viewpoints_candidate/testing_data/training_second_square_flange
Found 71 rows for feature 'surface6'.
Chamfer Distance range: -1.00 - 11.47 mm

Visualizing 71 viewpoint(s) for feature 'surface6'...


In [106]:
# ==========================================
# 5. VISUALIZE ENTIRE WORKPIECE & CAMERA POSES
# ==========================================
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt

print("Generating visualization of the entire workpiece and camera poses...")

# We use the full workpiece (vis_pcd) instead of filtering it!
geometries = [vis_pcd]

cmap = plt.get_cmap("tab10")

print("\n--- Selected Viewpoints Summary ---")
# For each step, create the camera pose arrows matching the step color
for k, v_idx in enumerate(selected_viewpoints, 1):
    step_color = cmap(k - 1)[:3]
    
    # Print the summary
    r, g, b = [int(c * 255) for c in step_color]
    colored_text = f"\033[38;2;{r};{g};{b}mStep {k}\033[0m"
    print(f"{colored_text}: Viewpoint {v_idx}")
    
    # Add the colored pose arrows to the visualization
    if v_idx in poses_dict:
        pose_arrows = make_xz_arrows(transform=poses_dict[v_idx], size=AXIS_SIZE, color=step_color)
        geometries.append(pose_arrows)

print("\nOpening Open3D visualization window...")

o3d.visualization.draw_geometries(
    geometries, 
    window_name="Entire Workpiece and Camera Poses",
    width=1024, height=768,
    front=[0, 0, 1], lookat=[0, 0, 0], up=[0, 1, 0], zoom=1.0
)


Generating visualization of the entire workpiece and camera poses...

--- Selected Viewpoints Summary ---
Step 1: Viewpoint 16
Step 2: Viewpoint 46
Step 3: Viewpoint 26
Step 4: Viewpoint 20
Step 5: Viewpoint 42
Step 6: Viewpoint 25
Step 7: Viewpoint 47
Step 8: Viewpoint 31

Opening Open3D visualization window...
